# 기자 질의 골든셋 v2.1 — 직접 확인 노트북

`eval/reporter-table-discovery-30/v2.1/`의 200문항과 채점기를 **눈으로 확인**하기 위한 노트북입니다. 확인용이라 Git에 올리지 않습니다(untracked이므로 `git clean -fd`를 돌리면 사라집니다).

**미리 알면 좋은 것**
- 채점은 `scripts/evaluate_runtime.py`가 합니다. 노트북에서 채점 로직을 복사하지 않고 그 실행기를 그대로 돌립니다.
- 채점 대상은 **표 탐색 경계**까지입니다. 수치 조회, CSV, 차트, 해석 문장은 채점하지 않습니다(수치·출력 호출은 0회여야 합니다).
- 골드의 정답 근거는 카탈로그 349표이고, `research/`는 질문 소재일 뿐입니다.
- 합격 기준: 경계/HTTP/미분류/골드의심 0, v2 원본 30/30, `resolved` ≥95%, `no_match` ≥90%, 역질문 ≥5건.

**실행 순서**: 커널은 아무거나 무관합니다(Anaconda 3.13도 됨). 5장이 리포 `.venv` 파이썬을 서브프로세스로 실행합니다.
1~4장은 키 없이 골드만 확인하고, 5장부터 실제 API를 호출하므로 먼저 `LIMIT = 5`로 확인한 뒤 `LIMIT = None`으로 전체를 돌리세요.


## 1. 경로 설정 · 임포트

노트북 위치가 어디든 리포 루트를 찾아 v2.1 `scripts/`를 불러옵니다.


In [ ]:
from pathlib import Path
import importlib.util, json, sys


def find_root(start):
    p = Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / "eval/reporter-table-discovery-30/v2.1/scripts/evaluate_runtime.py").is_file():
            return cand
    raise FileNotFoundError("StatBridge 리포 루트를 찾지 못했습니다. 리포 안에서 실행하세요.")


ROOT = find_root(Path.cwd())
V21 = ROOT / "eval/reporter-table-discovery-30/v2.1"
sys.path[:0] = [str(ROOT / "src/agent"), str(ROOT / "src/backend")]


def load(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


R = load("reporter200_runner", V21 / "scripts/evaluate_runtime.py")   # 실행기 + 채점기
V = load("reporter200_validate", V21 / "scripts/validate.py")        # 골드 검증기

CASES = R.load_cases()
MANIFEST = json.loads((V21 / "dataset_manifest.json").read_text(encoding="utf-8"))
CATALOG = json.loads((ROOT / "src/agent/stat_dictionary/stat_language_dictionary.json").read_text(encoding="utf-8-sig"))

VENV = ROOT / ".venv"
VENV_PY = VENV / "Scripts/python.exe" if (VENV / "Scripts/python.exe").exists() else VENV / "bin/python"
VENV_VERSION = ""
if (VENV / "pyvenv.cfg").exists():
    for line in (VENV / "pyvenv.cfg").read_text(encoding="utf-8", errors="replace").splitlines():
        if line.strip().startswith("version"):
            VENV_VERSION = line.split("=", 1)[1].strip()

print("ROOT     :", ROOT)
print("문항     :", len(CASES))
print("정책     :", R.POLICY["version"], "|", R.policy_sha256()[:12])
print("데이터셋 :", R.text_sha256(V21 / "statbridge-reporter200.jsonl")[:12])
print("정책 해시가 manifest와 일치:", MANIFEST["scoring_policy_sha256"] == R.policy_sha256())
print("실행 파이썬:", VENV_PY, "| .venv 버전:", VENV_VERSION or "알 수 없음", "| 존재:", VENV_PY.exists())
print("지금 커널  :", sys.executable, sys.version.split()[0], "(커널은 아무거나 무관)")


## 2. 골드 현황


In [ ]:
from collections import Counter

NEW = [case for case in CASES if case["schema_version"] != R.LEGACY_SCHEMA]

print(f"{'상태':18} {'전체':>5} {'증강':>5}")
for status in ("resolved", "no_match", "need_clarification"):
    print(f"{status:18} {sum(R.status_of(c) == status for c in CASES):>5} {sum(R.status_of(c) == status for c in NEW):>5}")
print()
print("구성(전체):", dict(Counter(R.composition_of(c) or "거절·역질문" for c in CASES)))
print("주기(resolved):", dict(Counter(c["workflow_gold"]["query_interpretation"]["frequency"]
                                   for c in CASES if R.composition_of(c))))
print("역질문 그룹:", dict(Counter(c["workflow_gold"]["resolution"]["clarification"]["id"]
                                for c in CASES if R.status_of(c) == "need_clarification")))
print("정답 표:", len({p["table_id"] for c in CASES for p in c["workflow_gold"]["classification_and_plan"]["series"]}),
      "| 계열:", sum(c["workflow_gold"]["resolution"]["series_count"] for c in CASES))
print("research 출처:", len({ref["figure_id"] for c in CASES for ref in c.get("research_references") or []}))
print()
print("--- 문항 목록(앞 20건) ---")
for case in CASES[:20]:
    tables = sorted({p["table_id"] for p in case["workflow_gold"]["classification_and_plan"]["series"]})
    print(f"{case['id']:22} {R.status_of(case):18} {','.join(tables) or '-':32} {case['query'][:44]}")


## 3. 골드 자체 검사 (런타임 없이)

검증기가 카탈로그 코드·기간·API 파라미터·v2 원본 보존·research 출처 해시를 확인합니다.


In [ ]:
summary = V.validate_cases(CASES, CATALOG, MANIFEST["counts"]["cases"])
V.validate_legacy_preservation(CASES)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("manifest와 일치:", summary == MANIFEST["counts"])


### 3-1. 채점 계약 자체 점검

골드에서 **완벽한 런타임 응답**을 합성해 채점합니다. 200/200이 아니면 채점기와 골드가 어긋난 것이고, 그 문항은 애초에 통과할 수 없습니다.


In [ ]:
def perfect_prediction(case):
    workflow = case["workflow_gold"]
    status = workflow["resolution"]["agent_status"]
    plans = workflow["classification_and_plan"]["series"]
    body = {"status": workflow["resolution"]["api_status"]}
    if status == "need_clarification":
        body["clarifications"] = [workflow["resolution"]["clarification"]]
    if status == "resolved":
        body["periodSelection"] = ({} if workflow["post_discovery"]["period_selection_required"]
                                   else {"source": "text"})
    return {"id": case["id"], "repeat_of": None, "query": case["query"], "http_status": 200,
            "agent_status": status, "api_status": workflow["resolution"]["api_status"],
            "selected_table_ids": sorted({p["table_id"] for p in plans}), "api_plans": plans,
            "boundary_response": body, "numeric_calls": 0, "output_calls": 0, "duration_s": 1.0, "error": None}


echo_scored = [R.score_case(case, perfect_prediction(case)) for case in CASES]
echo_summary = R.summarize(echo_scored, [], CASES, {"numeric_calls": 0, "output_calls": 0})
print("완벽 예측 채점:", echo_summary["passed"], "/", echo_summary["cases"])
print("판정:", R.decide(echo_summary, []))
print("실패 문항:", [(row["id"], row["failed_checks"]) for row in echo_scored if not row["passed"]] or "없음")
print()
print("--- 체크별 적용 문항 ---")
for check, stats in echo_summary["check_results"].items():
    print(f"  {check:22} {stats['passed']}/{stats['total']}")


## 4. 문항 눈으로 보기

`CASE_ID`를 바꾸면 다른 문항을 볼 수 있습니다. 골드의 계열·분류·기간·정확한 API 파라미터·출처를 그대로 출력합니다.


In [ ]:
CASE_ID = "REPORTER30-V2.1-037"          # 다른 문항 id로 바꿔 보세요
case = next(item for item in CASES if item["id"] == CASE_ID)
workflow = case["workflow_gold"]
print("질문:", case["query"])
print("상태:", R.status_of(case), "| API 경계:", workflow["resolution"]["api_status"])
print("기간:", workflow["query_interpretation"]["period"], "| 주기:", workflow["query_interpretation"]["frequency"])
for plan in workflow["classification_and_plan"]["series"]:
    print(f"  [{plan['table_id']}] {plan['table_name']} | {plan['label']}")
    print(f"     분류: {plan['classification_labels']} | 기간: {plan['period_start']}~{plan['period_end']}")
    print(f"     파라미터: {plan['exact_params']}")
for ref in case.get("research_references") or []:
    print("출처:", ref["path"], "|", ref["line"], "행 |", ref["heading"])
print()
print(json.dumps(case, ensure_ascii=False, indent=2)[:1200], "...")


## 5. 실행 방식 (커널과 무관)

채점은 **리포 `.venv` 파이썬을 서브프로세스로 실행**합니다. 지금 쓰는 커널(Anaconda 3.13)이 `.venv` 3.11과 달라도 동작합니다.
`.venv`에는 `fastapi`·`langgraph`·`plotly`·`pandas`와 `.env` 키가 이미 있어서 별도 설정이 필요 없습니다.

- `LIMIT`: 앞 몇 건만 실행해 배선만 확인합니다(기본 5, 전체는 `None`).
- `RESUME`: 커널이나 창이 죽은 뒤 이어서 실행할 때 `True`. 정책·데이터셋 해시가 다르면 섞지 않고 중단합니다.


In [ ]:
import subprocess, time

RUNNER = V21 / "scripts" / "evaluate_runtime.py"
OUT_DIR = ROOT / "evaluation_runs" / "reporter200_v2.1_notebook"
LIMIT = 5          # None이면 전체 220회(200문항 + 반복 20건)
RESUME = False     # 중단 후 이어서 돌릴 때 True
# 골드 의심 목록은 7장에서 지정합니다(재판정은 7장 셀에서 계산).

PREFLIGHT = ("import pathlib, sys\n"
             "sys.path.insert(0, r'%s')\n"
             "import fastapi, langgraph, plotly, pandas\n"
             "from authenticated_client import TestClient\n"
             "print('stack ok')\n"
             "print('auth ok')\n"
             "print('runner', pathlib.Path(sys.argv[1]).exists())\n"
             "print('gold', pathlib.Path(sys.argv[2]).exists())\n") % (ROOT / "tests")

probe = subprocess.run([str(VENV_PY), "-c", PREFLIGHT, str(RUNNER), str(V21 / "statbridge-reporter200.jsonl")],
                       cwd=ROOT, capture_output=True, text=True)
report = (probe.stdout + probe.stderr).strip()
print(report or "(사전 점검 출력 없음)")
if probe.returncode != 0 or "runner True" not in report or "gold True" not in report or "auth ok" not in report:
    print()
    print("리포 .venv 파이썬으로 실행할 수 없습니다:", VENV_PY)
    print("Windows 경로(C:\...)에서 노트북을 열었는지 확인하세요. WSL 경로(/mnt/c/...)는 Windows 파이썬이 읽지 못합니다.")
    print("'auth ok'가 없으면 인증 클라이언트(tests/authenticated_client.py)를 불러오지 못한 것이고, 그대로 실행하면 401이 납니다.")
else:
    cmd = [str(VENV_PY), str(RUNNER), "--output-dir", str(OUT_DIR)]
    if LIMIT:
        cmd += ["--limit", str(LIMIT)]
    if RESUME:
        cmd += ["--resume"]
    print()
    print("실행:", " ".join(cmd))
    print("(진행 로그와 예상 잔여 시간이 아래에 그대로 출력됩니다. 오래 걸립니다.)", flush=True)
    started = time.time()
    completed = subprocess.run(cmd, cwd=ROOT)
    print()
    print(f"종료 코드 {completed.returncode} | 총 {time.time() - started:.1f}초")


## 6. 결과 해석

5장에서 만든 `scores.json`을 읽어 채점 결과를 정리합니다. (`LIMIT` 부분 실행은 배선 확인용이며 점수가 아닙니다)


In [ ]:
scores_path = OUT_DIR / "scores.json"
if not scores_path.exists():
    print("아직 실행 결과가 없습니다. 5장 실행 셀을 먼저 돌리세요.")
    SUMMARY = None
else:
    SUMMARY = json.loads(scores_path.read_text(encoding="utf-8"))
    print("판정:", SUMMARY.pop("decision"))
    SUMMARY.pop("policy_sha256", None)
    print(f"전체 {SUMMARY['passed']}/{SUMMARY['cases']} | 실패 {SUMMARY['failed']} | "
          f"경계 위반 {SUMMARY['boundary_violations']} | 미분류 {SUMMARY['unclassified_failures']} | HTTP 실패 {SUMMARY['http_failures']}")
    print(f"v2 원본 {SUMMARY['legacy']['passed']}/{SUMMARY['legacy']['cases']} | "
          f"no_match 재현율 {SUMMARY['no_match_recall']} | 정밀도 {SUMMARY['no_match_precision']}")
    print(f"반복 일치 {SUMMARY['repeat_agreement']['agree']}/{SUMMARY['repeat_agreement']['cases']} | "
          f"총 {SUMMARY['timing']['total_seconds']}s | P50 {SUMMARY['timing']['p50']}s | P95 {SUMMARY['timing']['p95']}s")
    print()
    for title, block in (("상태별", "by_kind"), ("구성별", "by_composition"), ("주기별", "by_frequency"),
                         ("체크별", "check_results")):
        print(f"--- {title} ---")
        for name, stats in SUMMARY[block].items():
            print(f"  {str(name):22} {stats['passed']}/{stats['cases'] if 'cases' in stats else stats['total']}")
    print("--- 상태 혼동행렬 (기대|실제) ---")
    for key, count in SUMMARY["status_confusion"].items():
        print(f"  {key:42} {count}")
    print("--- 실패 분류 ---")
    print("  ", SUMMARY["failures"] or "없음")
    print()
    print("진단 리포트:", OUT_DIR / "diagnostic.md")


### 6-1. 실패 문항 드릴다운


In [ ]:
if SUMMARY is None:
    print("6장 요약 셀을 먼저 돌리세요.")
else:
    # scores.json에는 문항별 체크가 없으므로 실행 폴더의 문항 파일을 다시 채점합니다.
    failed = []
    for path in sorted(OUT_DIR.glob("REPORTER30-*.json")):
        if path.name.endswith(".repeat.json"):
            continue
        record = json.loads(path.read_text(encoding="utf-8"))
        gold_case = next(item for item in CASES if item["id"] == record["id"])
        row = R.score_case(gold_case, record)
        if not row["passed"]:
            failed.append({"id": row["id"], "row": row, "record": record, "gold": gold_case})
    print("실패 문항:", len(failed))
    for item in failed[:30]:
        row = item["row"]
        print(f"  {row['id']:22} {row['expected_status']}->{row['actual_status']} "
              f"{row['failed_checks']} | error={row['error']}")
    if len(failed) > 30:
        print(f"  ... 외 {len(failed) - 30}건 (diagnostic.md 참고)")

    FAIL_ID = failed[0]["id"] if failed else None    # 확인할 문항 id
    if FAIL_ID:
        item = next(entry for entry in failed if entry["id"] == FAIL_ID)
        row, record, gold_case = item["row"], item["record"], item["gold"]
        print()
        print("질문:", gold_case["query"])
        print("체크:", json.dumps(row["checks"], ensure_ascii=False))
        print("기대 표:", row["expected_table_ids"], "| 실제 표:", row["actual_table_ids"])
        for expected in gold_case["workflow_gold"]["classification_and_plan"]["series"]:
            print(f"  기대 계열: {expected['table_id']} {expected['classification_labels']} "
                  f"{expected['period_start']}~{expected['period_end']}")
        for actual in record.get("api_plans") or []:
            print(f"  실제 계열: {actual.get('table_id')} {actual.get('classification_labels')} "
                  f"{actual.get('period_start')}~{actual.get('period_end')}")
        print("원문 상태:", record.get("agent_status"), "/", record.get("api_status"),
              "| 숫자 호출:", record.get("numeric_calls"), "| 오류:", record.get("error"))
        print("원문 응답:", json.dumps(record.get("boundary_response"), ensure_ascii=False)[:600])


## 7. 골드 의심 표시

에이전트 잘못인지 골드 잘못인지 애매한 문항을 적으면 그 실행은 합격으로 저장되지 않습니다.


In [ ]:
if SUMMARY is None:
    print("6장 요약 셀을 먼저 돌리세요.")
else:
    import shutil

    SUSPECT = []      # 예: ["REPORTER30-V2.1-142"]
    (OUT_DIR / "gold_suspect.json").write_text(
        json.dumps(sorted(SUSPECT), ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print("의심 문항:", SUSPECT or "없음")
    print("재판정:", R.decide(SUMMARY, SUSPECT))
    if SUSPECT:
        if (OUT_DIR / "REPORT.md").exists():
            shutil.move(OUT_DIR / "REPORT.md", OUT_DIR / "REPORT.withdrawn.md")
            print("골드 의심이 있어 REPORT.md를 REPORT.withdrawn.md로 옮겼습니다(합격 표시를 남기지 않습니다).")
        print("정식 재판정은 아래 명령으로(의심 목록을 반영해 다시 계산합니다):")
        print(f"  .\\.venv\\Scripts\\python.exe {RUNNER.relative_to(ROOT)} "
              f"--output-dir {OUT_DIR.relative_to(ROOT)} --resume --gold-suspect-file "
              f"{(OUT_DIR / 'gold_suspect.json').relative_to(ROOT)}")


## 8. 산출물과 한계

**산출물** (`evaluation_runs/reporter200_v2.1_notebook/`)
`query_only_inputs.jsonl`(골드 미유입 증명) · `environment.json` · `endpoint_logs.jsonl` · 문항별 `<id>.json` · `scores.json` · `diagnostic.md` · `run_manifest.json` · 합격 시 `REPORT.md`

**채점하지 않는 것**
- KOSIS 수치 정확성, 로컬 CSV 값, 차트·그래프, 해석 문장과 인과 주장
- `no_match_reason`·`unmatched_concepts` 문구, 역질문 UI 문구

**한계**
- `LIMIT` 부분 실행은 배선 확인용이며 점수가 아닙니다. 정식 판정은 220회 전체 실행으로만 합니다.
- 반복 표본은 상태·구성만 층화하고 주기(M/Q/A) 대표성은 보장하지 않습니다.
- `seasonal_adjustment` 역질문 그룹은 카탈로그 `trigger_terms`가 M2 관련어라 실질 GDP 문항에서 런타임이 역질문을 띄우지 않을 수 있습니다.
- `no_match` 45건의 부재 판정은 공개 카탈로그 검색에 근거하며 KOSIS 실시간 호출로 재확인하지 않았습니다.
- 자동 채점 통과는 골드의 사람 승인을 뜻하지 않습니다.
